# 04 · OOD Generalization on Twitter

Evaluates all five models on the Twitter Financial News Sentiment dataset
without any retraining. This tests how well each approach generalises from
formal financial sentences (PhraseBank) to informal social media text (Twitter).

**Requires:** Checkpoints from notebook 02 and cached LLM responses from notebook 03.

**Writes:** `results/generalization.json`

In [ ]:
!pip install -q 'datasets>=2.14,<4.0' transformers torch scikit-learn anthropic python-dotenv tenacity nest_asyncio numpy

## 1 · Configuration

In [ ]:
# ── Toggle this when running on Colab ──────────────────────────
USE_DRIVE  = False
DRIVE_PATH = '/content/drive/MyDrive/cs443-final-project'
# ───────────────────────────────────────────────────────────────

import asyncio, hashlib, json, os, re, random, time
import numpy as np
from pathlib import Path
import nest_asyncio
nest_asyncio.apply()

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path(DRIVE_PATH)
else:
    BASE = Path('.')

RESULTS_DIR = BASE / 'results'
MODELS_DIR  = BASE / 'models'
CACHE_DIR   = BASE / 'cache'
for d in [RESULTS_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# API key for LLM evaluation
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

import anthropic
client = anthropic.Anthropic()
print('Base path:', BASE.resolve())

## 2 · Label Scheme & Metrics Helpers

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

ID2LABEL    = {0: 'negative', 1: 'neutral', 2: 'positive'}
LABEL2ID    = {v: k for k, v in ID2LABEL.items()}
LABEL_NAMES = ['negative', 'neutral', 'positive']


def compute_metrics(y_true, y_pred):
    y_true, y_pred = list(y_true), list(y_pred)
    acc      = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average='macro',
                               labels=[0, 1, 2], zero_division=0))
    per_f1   = f1_score(y_true, y_pred, average=None,
                        labels=[0, 1, 2], zero_division=0)
    cm       = confusion_matrix(y_true, y_pred, labels=[0, 1, 2]).tolist()
    return {
        'accuracy':         acc,
        'macro_f1':         macro_f1,
        'per_class_f1':     {ID2LABEL[i]: float(per_f1[i]) for i in range(3)},
        'confusion_matrix': cm,
    }


def print_metrics(m, title=''):
    if title:
        print('\n' + '─' * 54)
        print('  ' + title)
        print('─' * 54)
    acc = m['accuracy']
    f1  = m['macro_f1']
    print(f'  Accuracy : {acc:.4f}   Macro F1: {f1:.4f}')
    for name, val in m['per_class_f1'].items():
        print(f'  {name:<10}: F1={val:.4f}')

## 3 · Load Twitter OOD Set

In [ ]:
from datasets import load_dataset

TWITTER_REMAP = {0: 0, 1: 2, 2: 1}   # Bearish->neg, Bullish->pos, Neutral->neu

print('Loading Twitter Financial News Sentiment...')
raw_tw  = load_dataset('zeroshot/twitter-financial-news-sentiment')
tw_full = raw_tw['validation']
tw_full = tw_full.map(lambda ex: {'label': TWITTER_REMAP[ex['label']]})

tw_labels_arr = np.array(tw_full['label'])
rng = np.random.default_rng(SEED)
indices = []
for cls in range(3):
    cls_idx = np.where(tw_labels_arr == cls)[0]
    n = round(1000 * len(cls_idx) / len(tw_labels_arr))
    indices.extend(rng.choice(cls_idx, size=n, replace=False).tolist())
rng.shuffle(indices)
tw_ds = tw_full.select(indices[:1000])

tw_texts  = list(tw_ds['text'])
tw_labels = list(tw_ds['label'])

counts = {ID2LABEL[i]: tw_labels.count(i) for i in range(3)}
print(f'  Twitter OOD: {len(tw_texts)} examples | {counts}')

## 4 · TF-IDF + LogReg on Twitter

Retrain on PhraseBank train split (fast, <5 s) then predict on Twitter OOD.

In [ ]:
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Rebuild PhraseBank train split (same seed → identical to notebook 01)
raw_pb = load_dataset('gtfintechlab/financial_phrasebank_sentences_allagree')
ds_pb  = raw_pb['train']
if 'sentence' in ds_pb.column_names:
    ds_pb = ds_pb.rename_column('sentence', 'text')

pb_texts  = list(ds_pb['text'])
pb_labels = list(ds_pb['label'])
train_texts, _, train_labels, _ = train_test_split(
    pb_texts, pb_labels, test_size=0.30, random_state=SEED, stratify=pb_labels
)

vec = TfidfVectorizer(max_features=20_000, ngram_range=(1, 2), sublinear_tf=True)
X_train = vec.fit_transform(train_texts)
X_tw    = vec.transform(tw_texts)

clf = LogisticRegression(max_iter=1000, random_state=SEED, solver='lbfgs')
clf.fit(X_train, train_labels)
y_pred = clf.predict(X_tw).tolist()

tfidf_ood = compute_metrics(tw_labels, y_pred)
tfidf_ood['num_examples'] = len(tw_labels)
tfidf_ood['cost_per_1k']  = 0.0
print_metrics(tfidf_ood, 'TF-IDF + LogReg — Twitter OOD')

## 5 · Fine-tuned Transformers on Twitter

Loads the full-training checkpoints saved by notebook 02.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer

MAX_LEN = 128


def eval_checkpoint_on(ckpt_dir, texts, true_labels, title):
    ckpt_dir = Path(ckpt_dir)
    if not ckpt_dir.exists():
        print(f'  SKIP: checkpoint not found at {ckpt_dir}')
        return None

    print(f'  Loading checkpoint: {ckpt_dir}')
    tokenizer = AutoTokenizer.from_pretrained(str(ckpt_dir))
    model     = AutoModelForSequenceClassification.from_pretrained(str(ckpt_dir))

    from datasets import Dataset
    ds = Dataset.from_dict({'text': texts, 'label': true_labels})
    tok = ds.map(
        lambda b: tokenizer(b['text'], padding='max_length',
                            truncation=True, max_length=MAX_LEN),
        batched=True, remove_columns=['text'],
    )
    tok.set_format('torch')

    trainer = Trainer(model=model)
    t0  = time.perf_counter()
    out = trainer.predict(tok)
    elapsed = time.perf_counter() - t0

    y_pred = np.argmax(out.predictions, axis=-1).tolist()
    m = compute_metrics(true_labels, y_pred)
    m['num_examples']      = len(true_labels)
    m['inference_seconds'] = round(elapsed, 4)
    m['cost_per_1k']       = 0.0
    print_metrics(m, title)
    return m


roberta_ood = eval_checkpoint_on(
    MODELS_DIR / 'roberta-base_phrasebank',
    tw_texts, tw_labels,
    'RoBERTa-base — Twitter OOD',
)

finbert_ood = eval_checkpoint_on(
    MODELS_DIR / 'finbert_phrasebank',
    tw_texts, tw_labels,
    'FinBERT — Twitter OOD',
)

## 6 · LLM Zero-Shot on Twitter

Uses a separate cache file (`llm_<model>_twitter_ood.jsonl`) so cached
PhraseBank responses from notebook 03 are not mixed in.

In [ ]:
from tenacity import (
    retry, retry_if_exception_type, stop_after_attempt, wait_exponential
)

MODELS = {
    'haiku':  'claude-haiku-4-5-20251001',
    'sonnet': 'claude-sonnet-4-6',
}
PRICING = {
    'claude-haiku-4-5-20251001': {'input': 0.80,  'output': 4.00},
    'claude-sonnet-4-6':         {'input': 3.00,  'output': 15.00},
}

SYSTEM_PROMPT = 'You are a financial sentiment classifier.'
USER_TEMPLATE = (
    'Classify the following sentence from the perspective of an investor '
    'as positive, negative, or neutral.\n\n'
    'Sentence: {sentence}\n\n'
    'Respond with only one tag: <answer>positive</answer>, '
    '<answer>negative</answer>, or <answer>neutral</answer>.'
)


def cache_key(text):
    return hashlib.sha256(text.encode()).hexdigest()

def load_cache(path):
    cache = {}
    p = Path(path)
    if p.exists():
        with open(p) as f:
            for line in f:
                e = json.loads(line)
                cache[e['key']] = e
    return cache

def append_cache(path, entry):
    with open(path, 'a') as f:
        f.write(json.dumps(entry) + '\n')

def parse_answer(text):
    m = re.search(r'<answer>\s*(positive|negative|neutral)\s*</answer>',
                  text, re.IGNORECASE)
    return m.group(1).lower() if m else None


@retry(
    retry=retry_if_exception_type((
        anthropic.RateLimitError, anthropic.InternalServerError,
        anthropic.APIConnectionError,
    )),
    wait=wait_exponential(multiplier=1, min=2, max=60),
    stop=stop_after_attempt(5),
)
def _call_api_sync(model_id, text):
    return client.messages.create(
        model=model_id, max_tokens=32, temperature=0,
        system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': USER_TEMPLATE.format(sentence=text)}],
    )


async def evaluate_llm_ood(model_id, texts, true_labels, max_workers=10):
    cache_file = CACHE_DIR / f'llm_{model_id}_twitter_ood.jsonl'
    cache = load_cache(cache_file)
    predictions = [None] * len(texts)
    token_in = token_out = 0
    sem = asyncio.Semaphore(max_workers)

    async def process(i, text):
        nonlocal token_in, token_out
        key = cache_key(text)
        if key in cache:
            entry  = cache[key]
            parsed = entry.get('parsed_label')
            if parsed and parsed in LABEL2ID:
                predictions[i] = LABEL2ID[parsed]
                token_in  += entry.get('input_tokens', 0)
                token_out += entry.get('output_tokens', 0)
                return
        async with sem:
            response = await asyncio.to_thread(_call_api_sync, model_id, text)
        response_text = response.content[0].text
        parsed = parse_answer(response_text)
        entry = {
            'key': key, 'text': text, 'model': model_id, 'dataset': 'twitter_ood',
            'response_text': response_text, 'parsed_label': parsed,
            'input_tokens': response.usage.input_tokens,
            'output_tokens': response.usage.output_tokens,
        }
        append_cache(cache_file, entry)
        cache[key] = entry
        token_in  += response.usage.input_tokens
        token_out += response.usage.output_tokens
        predictions[i] = LABEL2ID.get(parsed, LABEL2ID['neutral'])

    await asyncio.gather(*[process(i, t) for i, t in enumerate(texts)])

    m       = compute_metrics(true_labels, predictions)
    pricing = PRICING.get(model_id, {'input': 0, 'output': 0})
    cost    = (token_in * pricing['input'] + token_out * pricing['output']) / 1_000_000
    m.update({
        'num_examples':       len(texts),
        'total_input_tokens': token_in, 'total_output_tokens': token_out,
        'estimated_cost_usd': round(cost, 4),
        'cost_per_1k':        round(cost / len(texts) * 1000, 4) if texts else 0,
    })
    return m

In [ ]:
print('Evaluating Haiku on Twitter OOD...')
haiku_ood = asyncio.run(
    evaluate_llm_ood(MODELS['haiku'], tw_texts, tw_labels)
)
print_metrics(haiku_ood, 'Claude Haiku 4.5 — Twitter OOD')

print('\nEvaluating Sonnet on Twitter OOD...')
sonnet_ood = asyncio.run(
    evaluate_llm_ood(MODELS['sonnet'], tw_texts, tw_labels)
)
print_metrics(sonnet_ood, 'Claude Sonnet 4.6 — Twitter OOD')

## 7 · Save Generalization Results

In [ ]:
gen_results = {
    'tfidf_logreg': tfidf_ood,
    'roberta-base': roberta_ood,
    'finbert':      finbert_ood,
    'haiku':        haiku_ood,
    'sonnet':       sonnet_ood,
}

# Remove None entries (models whose checkpoints were missing)
gen_results = {k: v for k, v in gen_results.items() if v is not None}

gen_path = RESULTS_DIR / 'generalization.json'
with open(gen_path, 'w') as f:
    json.dump(gen_results, f, indent=2)
print('Saved:', gen_path)

print('\nOOD Summary (Twitter):')
for name, m in gen_results.items():
    acc = m['accuracy']
    f1  = m['macro_f1']
    print(f'  {name:<16}  acc={acc:.4f}  macro_f1={f1:.4f}')